# Setup and Starter Code

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from keras import layers, models, callbacks
from keras.datasets import cifar10
from keras.utils import to_categorical
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import time
# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)
# CIFAR-10 Class Names
CLASS_NAMES = ['airplane','automobile','bird','cat','deer',
'dog','frog','horse','ship','truck']
# Load Data
(x_train_full, y_train_full), (x_test, y_test) = cifar10.load_data()
# Fixed Split: 40,000 train / 10,000 val / 10,000 test
x_train = x_train_full[:40000].astype('float32')
y_train = y_train_full[:40000]
x_val = x_train_full[40000:].astype('float32')
y_val = y_train_full[40000:]
x_test = x_test.astype('float32')
print(f"Train: {x_train.shape} | Val: {x_val.shape} | Test:{x_test.shape}") 
print(f"Pixel range: [{x_train.min()}, {x_train.max()}]")
print(f"Image shape: {x_train[0].shape}")

C:\Users\matth\Downloads\College\Third year_term 2\supervised\assignment 2\CIFAR10-CNN-Optimization-main\venv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train: (40000, 32, 32, 3) | Val: (10000, 32, 32, 3) | Test:(10000, 32, 32, 3)
Pixel range: [0.0, 255.0]
Image shape: (32, 32, 3)


# 🟢 Task 1: Data Preprocessing Experiments

## 🔴 1A. Normalization Comparison

### ⚫ Description: Comparing No Normalization, Min-Max, and Standardization.

In [3]:
from tensorflow import keras
def BaselineCNN():
  model = models.Sequential([
      layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
      layers.MaxPooling2D((2, 2)),
      layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
      layers.MaxPooling2D((2, 2)),
      layers.Flatten(),
      layers.Dense(128, activation='relu'),
      layers.Dense(10, activation='softmax')
  ])
  return model

# EXP A
x_train_A = x_train.copy()
x_val_A = x_val.copy()
x_test_A = x_test.copy()

# EXP B
x_train_B = x_train / 255.0
x_val_B = x_val / 255.0
x_test_B = x_test / 255.0

# EXP C
mean = x_train.mean(axis=(0, 1, 2)) 
std = x_train.std(axis=(0, 1, 2))

x_train_C = (x_train - mean) / std
x_val_C = (x_val - mean) / std
x_test_C = (x_test - mean) / std

# Training Config
EPOCHS = 20
BATCH_SIZE = 128

# Dictionary to store different preprocessing versions of the dataset
experiments = {
    'Raw': (x_train_A, x_val_A,  x_test_A),
    'Min-Max': (x_train_B, x_val_B,  x_test_B),  
    'Standardized': (x_train_C, x_val_C,  x_test_C),
}

histories = {}
# Loop over each preprocessing method
for name, (x_tr, x_v, x_te) in experiments.items():
    model = BaselineCNN()
    model.compile(
        optimizer=keras.optimizers.Adam(0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    history = model.fit(
        x_tr, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(x_v, y_val),
        verbose=1
    )
    
    # Save training history (used later for plotting curves)
    histories[name] = history
    loss, acc = model.evaluate(x_te, y_test, verbose=0)
    print(f"{name} → Test Acc: {acc*100:.2f}%  |  Loss: {loss:.4f}")


# Plot 1: Training Loss
plt.figure(figsize=(10, 5))
for name, h in histories.items():
    plt.plot(h.history['loss'], label=name)
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.xticks(range(0, EPOCHS, 2))
plt.show()

# Plot 2: Validation Accuracy
plt.figure(figsize=(10, 5))
for name, h in histories.items():
    plt.plot(h.history['val_accuracy'], label=name)
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.xticks(range(0, EPOCHS, 2))
plt.show()

C:\Users\matth\Downloads\College\Third year_term 2\supervised\assignment 2\CIFAR10-CNN-Optimization-main\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 20ms/step - accuracy: 0.3857 - loss: 3.6788 - val_accuracy: 0.4754 - val_loss: 1.4882
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.5434 - loss: 1.2901 - val_accuracy: 0.5425 - val_loss: 1.3128
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.6060 - loss: 1.1212 - val_accuracy: 0.5533 - val_loss: 1.3199
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.6530 - loss: 0.9930 - val_accuracy: 0.5751 - val_loss: 1.2753
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.6877 - loss: 0.8907 - val_accuracy: 0.5822 - val_loss: 1.3038
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.7171 - loss: 0.8078 - val_accuracy: 0.5790 - val_loss: 1.3918
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7439 - loss: 0.7277 - val_accuracy: 0.5744 - val_loss: 1.5412
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7618 - loss: 0.6716 - val_accu

KeyboardInterrupt: 

## 🔴 1B. Data Augmentation Comparison

### ⚫ Description: Using the standardized data from Exp C, train the BaselineCNN with and without data augmentation

In [ ]:
# Training Config
EPOCHS = 40
BATCH_SIZE = 128

# Exp 1 : No Augmentation
model_withoutAug = BaselineCNN()
model_withoutAug.compile(optimizer=keras.optimizers.Adam(0.001),
                        loss='sparse_categorical_crossentropy',
                        metrics=['accuracy'])

history_withoutAug = model_withoutAug.fit(x_train_C, y_train,
                                         epochs=EPOCHS, batch_size=BATCH_SIZE,
                                         validation_data=(x_val_C, y_val), verbose=1)

loss_withoutAug, acc_withoutAug = model_withoutAug.evaluate(x_test_C, y_test, verbose=0)
print(f"No Augmentation → Test Acc: {acc_withoutAug*100:.2f}%  |  Loss: {loss_withoutAug:.4f}")

# Exp 2: With Augmentation 
datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1
)
datagen.fit(x_train_C)

model_withAug = BaselineCNN()
model_withAug.compile(optimizer=keras.optimizers.Adam(0.001),
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

history_withAug = model_withAug.fit(datagen.flow(x_train_C, y_train, batch_size=BATCH_SIZE),
                                    epochs=EPOCHS,
                                    validation_data=(x_val_C, y_val), verbose=1)

loss_withAug, acc_withAug = model_withAug.evaluate(x_test_C, y_test, verbose=0)
print(f"With Augmentation → Test Acc: {acc_withAug*100:.2f}%  |  Loss: {loss_withAug:.4f}")

# Plot 1: Training Loss
plt.figure(figsize=(10, 5))
plt.plot(history_withoutAug.history['loss'], label='No Augmentation')
plt.plot(history_withAug.history['loss'], label='With Augmentation')
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.xticks(range(0, EPOCHS, 2))
plt.legend()
plt.show()

# Plot 2: Validation Accuracy
plt.figure(figsize=(10, 5))
plt.plot(history_withoutAug.history['val_accuracy'], label='No Augmentation')
plt.plot(history_withAug.history['val_accuracy'], label='With Augmentation')
plt.title('Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.xticks(range(0, EPOCHS, 2))
plt.legend()
plt.show()

# 🟢 Task 2: CNN Architecture Experiments

## ⚫ Use standardized data for ALL experiments in this task.

## 🔴 2A. Filter Count Comparison

### ⚫ Description: Testing Small, Medium, and Large filter configurations.

In [14]:
import time
import pandas as pd

# Function to build models with diffrent filter counts
def build_model_2a(f1, f2, f3, f4):
    model = models.Sequential([
        layers.Conv2D(f1, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.Conv2D(f2, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(f3, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(f4, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

filter_configs = {
    'Small': (8, 8, 16, 16),
    'Medium': (32, 32, 64, 64),
    'Large': (64, 64, 128, 128)
}

histories_2a = {}
results_2a = []

for name, (f1, f2, f3, f4) in filter_configs.items():
    print(f"\n=================================")
    print(f"Training Model: {name} Filter Count")
    print(f"=================================")
    
    model = build_model_2a(f1, f2, f3, f4)
    model.summary()
    
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    # train model
    start_time = time.time()
    history = model.fit(
        x_train_C, y_train, 
        epochs=20, 
        batch_size=128, 
        validation_data=(x_val_C, y_val), 
        verbose=1
    )
    time_taken = time.time() - start_time
    
    histories_2a[name] = history
    
    # Evaluation
    test_loss, test_acc = model.evaluate(x_test_C, y_test, verbose=0)
    
    # Store metrics for table
    results_2a.append({
        'Model': name,
        'Total Params': model.count_params(),
        'Train Acc': history.history['accuracy'][-1],
        'Val Acc': history.history['val_accuracy'][-1],
        'Test Acc': test_acc,
        'Time (s)': time_taken
    })

# results Table
df_2a = pd.DataFrame(results_2a)
print(f"\n=================================")
print("Results: Filter Count Comparison")
print(f"=================================")
display(df_2a)


Training Model: Small Filter Count


Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_58 (Conv2D)                   │ (None, 32, 32, 8)           │             224 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_59 (Conv2D)                   │ (None, 32, 32, 8)           │             584 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_26 (MaxPooling2D)      │ (None, 16, 16, 8)           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_60 (Conv2D)                   │ (None, 16, 16, 16)          │           1,168 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_61 (Conv2D)                   │ (None, 16, 16, 16)          │           2,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_27 (MaxPooling2D)      │ (None, 8, 8, 16)            │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_8 (Flatten)                  │ (None, 1024)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_24 (Dense)                     │ (None, 256)                 │         262,400 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_25 (Dense)                     │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 269,266 (1.03 MB)

 Trainable params: 269,266 (1.03 MB)

...

 Total params: 2,360,138 (9.00 MB)

 Trainable params: 2,360,138 (9.00 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 32s 98ms/step - accuracy: 0.4954 - loss: 1.4086 - val_accuracy: 0.6330 - val_loss: 1.0654
Epoch 2/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 31s 100ms/step - accuracy: 0.6833 - loss: 0.9008 - val_accuracy: 0.7003 - val_loss: 0.8621
Epoch 3/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 106ms/step - accuracy: 0.7629 - loss: 0.6847 - val_accuracy: 0.7258 - val_loss: 0.8036
Epoch 4/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 33s 106ms/step - accuracy: 0.8137 - loss: 0.5371 - val_accuracy: 0.7250 - val_loss: 0.8682
Epoch 5/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 96ms/step - accuracy: 0.8519 - loss: 0.4214 - val_accuracy: 0.7234 - val_loss: 0.8847
Epoch 6/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.8835 - loss: 0.3301 - val_accuracy: 0.7275 - val_loss: 0.9771
Epoch 7/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 97ms/step - accuracy: 0.9111 - loss: 0.2555 - val_accuracy: 0.7220 - val_loss: 1.1366
Epoch 8/20
313/313 ━━━━━━━━━━━━━━━━━━━━ 30s 95ms/step - accuracy: 0.9276 - loss: 0.2038

KeyboardInterrupt: 

In [ ]:
# Plot val accuracy curves for all 3 on one graph
plt.figure(figsize=(10, 6))
for name, h in histories_2a.items():
    plt.plot(h.history['val_accuracy'], label=f'{name} Model', lw=2)

plt.title('Validation Accuracy - Filter Count Comparison', fontsize=14)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Validation Accuracy', fontsize=12)
plt.xticks(range(0, 20, 2))
plt.legend(fontsize=11)
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

## 🔴 2B. Network Depth Comparison

### ⚫ Description: Comparing Shallow (4 layers), Medium (6 layers), and Deep (8 layers).

In [ ]:
def build_shallow():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

def build_medium_depth():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

def build_deep():
    model = models.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.Dense(10, activation='softmax')
    ])
    return model

depth_models = {
    'Shallow': build_shallow(),
    'Medium': build_medium_depth(),
    'Deep': build_deep()
}

histories_2b = {}
results_2b = []

for name, model in depth_models.items():
    print(f"\n=================================")
    print(f"Training Model: {name} Depth")
    print(f"=================================")
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    
    start_time = time.time()
    history = model.fit(
        x_train_C, y_train, 
        epochs=20, 
        batch_size=128, 
        validation_data=(x_val_C, y_val), 
        verbose=1
    )
    time_taken = time.time() - start_time
    
    histories_2b[name] = history
    test_loss, test_acc = model.evaluate(x_test_C, y_test, verbose=0)
    
    results_2b.append({
        'Model': name,
        'Total Params': model.count_params(),
        'Train Acc': history.history['accuracy'][-1],
        'Val Acc': history.history['val_accuracy'][-1],
        'Test Acc': test_acc,
        'Time (s)': time_taken
    })

# Results Table
df_2b = pd.DataFrame(results_2b)
print(f"\n=================================")
print("Results Table 2B: Network Depth Comparison")
print(f"=================================")
display(df_2b)

In [ ]:
# Plot training loss and val loss (2 curves per plot, 3 separate plots)
for name, h in histories_2b.items():
    plt.figure(figsize=(8, 5))
    plt.plot(h.history['loss'], label='Training Loss', lw=2)
    plt.plot(h.history['val_loss'], label='Validation Loss', lw=2)
    
    plt.title(f'{name} Depth Model: Training & Validation Loss', fontsize=14)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.xticks(range(0, 20, 2))
    plt.legend(fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

# 🟢 Task 3: Regularization Experiments

## ⚫ Use the Medium model from Task 2A as the baseline for all experiments.

## 🔴 3A. Dropout Rate Comparison

### ⚫ Description: Comparing rates of 0.0, 0.25, and 0.5.

In [6]:
pass

## 🔴 3B. Early Stopping Comparison

### ⚫ Description: Comparing no ES vs. patience=5 vs. patience=10.

In [7]:
pass

# 🟢 Task 4: Optimizer Comparison

## ⚫ Use the Medium model from Task 2A. Use standardized data.

## 🔴 4A. Same Learning Rate — 5 Optimizers.

### ⚫ Description: SGD, Momentum, AdaGrad, RMSProp, and Adam.

### ⚫ Train with all 5 optimizers at learning_rate = 0.001, 30 epochs, batch_size=128 

In [8]:
pass

## 🔴 4B. Learning Rate Sensitivity — Adam

### ⚫ Description: Testing LR values: 0.0001, 0.001, 0.01. (30 epochs, batch_size=128)

In [9]:
pass

# 🟢 Task 5: Performance Evaluation

## 🔴 5A. Best Model Selection

In [10]:
pass

## 🔴 5B. Error Analysis

In [16]:
pass

# 🟢 Task 6: Transfer Learning

## 🔴 6A. Feature Extraction vs Fine-Tuning vs From Scratch

### ⚫ Step 1 — Resize CIFAR-10 for VGG16: 

In [11]:
pass

### ⚫ Model 1: From Scratch 

- Build the Medium CNN from Task 2A with input_shape=(48, 48, 3). Train 20 epochs, Adam(lr=0.001),  batch_size=128. 

In [12]:
pass

### ⚫ Model 2 — Feature Extraction (Frozen VGG16) 

- Load a pre-trained VGG16 model (trained on ImageNet) without the top classification layers. 

- Freeze ALL VGG16 layers so their weights do not update during training.

- Add your own classification head on top. 

In [13]:
pass

### ⚫ Model 3 — Fine-Tuning (Partial Unfreeze) 

- Load a pre-trained VGG16 model (same as Model 2). 

- Freeze all layers EXCEPT the last 4 layers — these will be re-trained on CIFAR-10.

- Add the same classification head (Flatten → Dense(256) → Dropout → Dense(10)). 

- Use a LOWER learning rate (1e-5) to avoid destroying the pre-trained weights. 

In [14]:
pass

### ⚫ Model 4 — Fine-Tuning (Higher Learning Rate)

- Same architecture and frozen layers as Model 3 (unfreeze last 4 VGG16 layers), but use a higher learning  rate (lr=0.001) instead of 1e-5.

In [15]:
pass